# Levelised cost of green ammonia by cell: global results

This notebook shows the results of a global campaign of the green-lory plant model as maps, bar
charts, histograms, tables and supply curves. Each run of the campaign designs and costs an islanded
plant, which makes ammonia from wind and PV, in every one-degree cell, and gives the levelised cost of
ammonia (LCOA) of each cell's plant. The notebook reads the merged result tables of the global stage
of `arc/submit_lory_sequence.sh` (`merged/global_run_results.csv`, checked by the merge and QA job,
`arc/qa.py`), copied from ARC to `results/campaigns/`. `run_store.csv` gives the folder of each run,
and `model/results.py` reads the tables by run id and draws the figures. The notebook builds and runs
nothing.

In [ ]:
import importlib

import matplotlib.pyplot as plt
import pandas as pd

import model.results as h

importlib.reload(h)
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.width", 160)

## Runs and cases of the campaign

Every run solves the same plant in each cell: wind turbines and fixed PV supply an electrolyser,
batteries, hydrogen and ammonia storage and a Haber-Bosch synthesis loop, with no grid connection,
over the 8,760 hours of 2019 in ERA5 weather. The plant is designed to make 1 Mt/yr of ammonia, and
its LCOA (`lcoa_eur_per_t`) is its annualised cost divided by its annual output, in EUR per tonne.
How much ammonia a cell can make at that cost is limited by the area the cell gives to the plant, 20 %
of its suitable land, and is the cell's production potential
(`scaled_design_max_gridless_onshore_ammonia_capacity_t`, t/yr).

A land run covers the cells with suitable land. A sea run (a run id ending in `_offshore`) puts the
same plant at sea, with wind turbines and no PV, in every cell that is at least 1 % sea, at the costs
of the cell's water depth (section "Plants at sea"). A coastal cell so holds a land plant and a sea
plant, each on its own area.

The four cases differ only in the costs that depend on the location:

- `base`: a 5 % cost of capital everywhere, capital costs without location factors, and water at a
  uniform price of 2 EUR/m³ (2020 euros);
- `site`: capital costs multiplied by the location factors of the run's site-cost table
  (`inputs/site_costs/`), which are remoteness (equipment by the distance to a port, installation by
  the travel time to a town), cold and slope on land, and remoteness and cold at sea; water is priced
  at its source plus a pipeline from the nearest source;
- `finance`: the cost of capital of the cell's country, from the low-carbon rates of Ameli et al.
  (2021), and 11.8 % in international waters;
- `all`: the site costs of `site` with the cost of capital of `finance`.

`base` leaves out every cost that depends on the location, so remote, cold or frozen sites can look
cheap in `base` and `finance`; `all`, with every one of these costs, gives the most complete picture.
The table lists the runs with the options `run_store.csv` records for each (the tokens are defined in
`GLOSSARY.md`), its status and whether its results are in the local copies.

In [ ]:
# Land runs of the four cases: run ids from runs.yaml (`green-lory list` shows the others).
# The sea run of each case has the same id with "_offshore".
RUNS = {
    "base": "gl_dea2050_wacc5_bflat_wflat_land20oct_fixed",
    "site": "gl_dea2050_wacc5_bspat_wspat_land20oct_fixed",
    "finance": "gl_dea2050_amelilc_bflat_wflat_land20oct_fixed",
    "all": "gl_dea2050_amelilc_bspat_wspat_land20oct_fixed",
}
SEA_RUNS = {case: f"{run_id}_offshore" for case, run_id in RUNS.items()}
h.runs_table({**RUNS, **{f"{case} at sea": run_id for case, run_id in SEA_RUNS.items()}})

In [ ]:
for run_id in [*RUNS.values(), *SEA_RUNS.values()]:
    if not h.output_dir(run_id).is_dir():
        print(f"{run_id} has no local copy; copy it from ARC with\n  {h.copy_command(run_id)}")
LAND = {case: h.load_surface(run_id) for case, run_id in RUNS.items()}
SEA = {case: h.load_surface(run_id) for case, run_id in SEA_RUNS.items()}

## Location factors of the cost

The cases differ by three location factors, mapped here as the runs used them, from their result
tables, each in its own colour. The first map shows `build_cost_multiplier`, the factor by which the
location raises the capital cost of the land plants of `site` (it is 1 in `base` and `finance`). It
is close to 1 in most of the world and rises with remoteness and cold: the median cell has 1.3, the
central Sahara and the Tibetan Plateau, far from towns, have 2 to 3, and a tenth of the cells have
more than 3.6, most of them in the north of Russia and Canada, in Alaska and in Greenland.

In [ ]:
LIMITS = (1, 4)  # ends of the colour scale; larger factors take the end colour
factor = LAND["site"]["build_cost_multiplier"]
h.plot_maps({"": LAND["site"]}, "build_cost_multiplier", title="Location factor on the capital cost of the land plants: site",
            label="Factor on capital cost (-)", limits=LIMITS, ramp=h.RAMPS["orange"],
            notes={"": h.summary_note(factor, decimals=2)})
plt.show()

The second map shows the cost of capital of `finance` (`interest_rate_overall`), which is set by
country and is the same for every technology of a plant. A sea plant takes the rate of the country
whose exclusive economic zone holds its cell, and 11.8 % in international waters. The finance table
gives one rate to each of the 16 regions for which Ameli et al. estimate the cost of capital; the
table under the maps lists them (`finance_region` of `h.cell_regions`), with their land cells: from
2.4 % in Japan and 4.2 % in western Europe to 9.2 % in Central and South America and 11.8 % in Africa
and Mexico.

In [ ]:
LIMITS = (2, 12)  # ends of the colour scale in per cent
finance = {f"{site} plants": plants.assign(interest_rate_pct=100 * plants["interest_rate_overall"])
           for site, plants in (("Land", LAND["finance"]), ("Sea", SEA["finance"]))}
h.plot_maps(finance, "interest_rate_pct", title="Cost of capital of the plants: finance", label="Cost of capital (%)",
            limits=LIMITS, ramp=h.RAMPS["aqua"])
plt.show()
regions = finance["Land plants"].merge(h.cell_regions(), on=["latitude", "longitude"]).groupby("finance_region")
regions.agg(cells=("interest_rate_pct", "size"), cost_of_capital_pct=("interest_rate_pct", "median")).round(1).sort_values("cost_of_capital_pct")

The third map shows the water cost of the land plants of `site` per tonne of ammonia
(`water_cost_eur_per_t`): the price of water at its source plus a pipeline from the nearest source,
times the plant's water use. It is highest in the deserts, far from rivers and lakes, but small
against the LCOA: a median of 6.6 EUR/t and at most 79 EUR/t, against 3.2 EUR/t at the uniform price
of `base` and `finance`. Sea plants desalinate sea water at the source price, with no pipeline.

In [ ]:
LIMITS = None  # ends of the colour scale in EUR/t, or None for the 2nd and 98th percentiles
water = LAND["site"]["water_cost_eur_per_t"]
h.plot_maps({"": LAND["site"]}, "water_cost_eur_per_t", title="Water cost of the land plants: site",
            label="Water cost (EUR/t of ammonia)", limits=LIMITS, ramp=h.RAMPS["blue"],
            notes={"": h.summary_note(water, "EUR/t", decimals=1)})
plt.show()

## Levelised cost of ammonia by case

The maps show the four cases one after the other, on one colour scale. Each cell shows the cheaper of
its land and sea plants among those that can make at least 1 Mt/yr (`h.lowest_feasible_plant`), the
plants that count towards the supply curves and the network model's suppliers; `SITES` shows the land
or the sea plants alone. The colours are those of viridis, reversed so that yellow is the cheapest,
on a linear scale from 250 to 2,000 EUR/t; plants above 2,000 EUR/t are grey, and cells where no plant
can make 1 Mt/yr are light grey. The box in each panel gives the number of land and sea plants shown
and their median LCOA. The lines under the maps count, for each case, the sea plants up to
2,000 EUR/t and those of them south of 60 degrees S, and the cells that hold a land plant and a sea
plant that can both make 1 Mt/yr, with those of them where the sea plant is the cheaper.

In [ ]:
CASES = list(RUNS)  # the cases shown, one panel each, from "base", "site", "finance" and "all"
SITES = "both"  # "both" (each cell's cheaper plant), "land" or "sea"
MINIMUM_MT = 1.0  # the plants shown can make at least this (Mt/yr); 0 shows every plant
LIMITS = (250, 2000)  # ends of the colour scale in EUR/t; plants above the upper end are grey
shown = {case: h.select_plants(LAND[case], SEA[case], SITES, MINIMUM_MT) for case in CASES}
cells = h.select_plants(LAND[CASES[0]], SEA[CASES[0]], SITES, 0.0)
h.plot_maps(shown, "lcoa_eur_per_t", label="Levelised cost of ammonia (EUR/t)", limits=LIMITS, ramp=h.LCOA_COLOURMAP,
            title=f"Levelised cost of ammonia by case: {h.SITES[SITES]}" + (f" that can make {MINIMUM_MT:g} Mt/yr" if MINIMUM_MT else ""),
            above=f"above {LIMITS[1]:,} EUR/t", background=cells, background_label=f"no plant that makes {MINIMUM_MT:g} Mt/yr",
            notes={case: h.site_note(frame) for case, frame in shown.items()})
plt.show()
for case, frame in shown.items():
    sea = frame[(frame["site"] == "sea") & (frame["lcoa_eur_per_t"] <= LIMITS[1])]
    two = h.compare_surfaces(*(plants[plants[h.CAPACITY_COLUMN] >= MINIMUM_MT * 1e6] for plants in (SEA[case], LAND[case])))
    print(f"{case}: {len(sea):,} sea plants up to {LIMITS[1]:,} EUR/t, {(sea['latitude'] <= -60).sum():,} of them south of 60 degrees S; "
          f"{len(two):,} cells with two plants, in {(two['lcoa_change_pct'] < 0).sum():,} of which the sea plant is the cheaper")

In `base`, the land plants that can make 1 Mt/yr cost a median 307 EUR/t and are almost all yellow,
the cheap end of the scale. The sea plants cost a median 1,070 EUR/t; 33,744 of them cost up to
2,000 EUR/t, the cheapest, in green, in the windy belts of the Southern Ocean, the North Atlantic
and the North Pacific, and 4,149 of these lie south of 60 degrees S, off Antarctica, since `base`
has no cost for cold or remoteness. `site` adds these costs: the median land plant rises to 374
EUR/t, the central Sahara and the interior of Australia turn green, and only 10,635 sea plants stay
up to 2,000 EUR/t, along the coasts and in parts of the Southern Ocean, because the remoteness
factors of a sea plant count the distance to the nearest port. `finance` changes the colours by
country instead: Africa turns green with its cost of capital of 11.8 %, while the United States,
Australia and China stay yellow, and at sea the exclusive economic zones of countries with low rates
stand out from the open ocean, at 11.8 %; 26,369 sea plants stay up to 2,000 EUR/t. `all` combines
the two: the median land plant costs 509 EUR/t, the central Sahara turns blue, the United States and
the Arabian Peninsula stay yellow, and only 5,590 sea plants stay up to 2,000 EUR/t, none of them
south of 60 degrees S.

In every case, where a cell holds two plants that can make 1 Mt/yr, the land plant is the cheaper,
so a coastal cell shows its sea plant only where its land cannot make 1 Mt/yr. Light grey covers
most of Europe, the boreal forests and tundra of Russia, Canada and Alaska, Greenland and the
tropical forests, where 20 % of the suitable land is too small for a plant of 1 Mt/yr, and patches
of the equatorial oceans, where the wind is so weak that a plant of 1 Mt/yr needs more turbines than
20 % of the cell's sea can hold.

The table lists the ten cheapest land plants of one case that can make at least 1 Mt/yr, with the
cell's mean elevation (`elevation_m`; below zero where much of the cell is sea), wind's share of the
plant's generating capacity and the cell's production potential.

In [ ]:
CASE = "base"  # "base", "site", "finance" or "all"
MINIMUM_MT = 1.0  # the table lists plants that can make at least this (Mt/yr); 0 lists every plant
h.cheapest_cells(LAND[CASE], n=10, minimum_mt_per_year=MINIMUM_MT)

In `base`, the cheapest land plants of all, at 225 to 241 EUR/t, are on the small sub-Antarctic
islands (the Crozet, Prince Edward, Kerguelen, Heard, Macquarie, Campbell and Auckland Islands) and the
islands of southern Chile, where wind supplies most of the plant's power; but most of them have too
little land to make 1 Mt/yr (`MINIMUM_MT = 0` lists them), and `base` has no cost for their
remoteness. The cheapest plants that can make 1 Mt/yr are on the high plateau of the Andes, the Puna
and the Altiplano of Argentina, Chile and Bolivia, at 3,600 to 4,900 m, where PV alone powers the plant
at 257 to 260 EUR/t; each cell can make 12 to 40 Mt/yr. `base` has no cost for building at altitude,
in the cold or far from towns, so the high plateaus rank as its cheapest sites; `site` and `all` move
them down (`CASE = "all"` lists the cheapest plants of `all`).

The next table gives the headline figures of every case, for the land and for the sea plants, and
the cost of the last tonne at two global production volumes: the LCOA of the plant that brings the
cumulative potential of the cheapest plants to that volume (`h.cost_at_supply`). Only plants that can
make at least 1 Mt/yr count towards a volume, as for the network model's suppliers.

In [ ]:
VOLUMES = (500, 2000)  # global production volumes (Mt/yr) at which the table gives the cost of the last tonne
h.summary_table(LAND, SEA, volumes=VOLUMES)

## What the location factors add to the cost, by region

The chart splits the mean LCOA of `all` by UN sub-region (M49, the United Nations Statistics
Division's standard regions, `un_sub_region` of `h.cell_regions`): the cost of `base`, and on it what
each location factor adds (`h.cost_factors`): construction and water, from `site`; the cost of
capital, from `finance`; and the part that the two give together, since the cost of capital of `all`
applies to the capital cost that its site costs have raised. The means are over the land plants that
can make 1 Mt/yr in every case, with the number of plants of each sub-region in its label; unlike
medians, means add up, so each bar ends at its sub-region's mean LCOA, marked by a black tick. A part
below zero, such as finance where the cost of capital is below 5 %, is drawn left of zero. M49 puts all
of Russia in Eastern Europe, and Greenland and Canada in Northern America with the United States. The
line under the chart gives the mean of each part over all the plants.

In [ ]:
MINIMUM_MT = 1.0  # the plants included can make at least this (Mt/yr) in every case; 0 includes every land plant
BY = "un_sub_region"  # "un_sub_region", "un_region" (M49) or "finance_region" (the 16 regions of the finance table)
factors = h.cost_factors(LAND["base"], LAND["site"], LAND["finance"], LAND["all"])
factors = factors[factors["potential_mt_per_year"] >= MINIMUM_MT]
h.plot_region_costs(factors, ["construction", "water", "finance", "interaction"], by=BY,
                    title=f"Mean levelised cost of ammonia by region: all, {len(factors):,} land plants")
plt.show()
print("Mean of each part (EUR/t):", ", ".join(f"{h.FACTOR_NAMES[name]} {value:,.0f}"
                                            for name, value in factors[list(h.FACTOR_NAMES)].mean().items()))

Over the 7,934 land plants, `base` costs a mean 315 EUR/t. Site costs add 96 EUR/t, nearly all of it
construction (86 EUR/t, against 10 EUR/t of water), finance adds 97 EUR/t, and the two together add
28 EUR/t more than separately, for a mean of 535 EUR/t in `all`.

By sub-region, `all` is cheapest in Southern Europe (339 EUR/t), where the cost of capital is below
5 %; Western Europe (377 EUR/t) and Northern Europe (678 EUR/t) have only 6 and 2 plants. Next come
South-eastern, Western and Southern Asia and Northern America, at 413 to 430 EUR/t. Construction
adds most in Eastern Europe, which in M49 includes Russia, in Northern Africa, and in Australia and
New Zealand, Eastern Asia and Central Asia, where many plants are far from towns or ports. Finance
adds most in Sub-Saharan Africa, Northern Africa and Latin America and the Caribbean, at 9.2 to 11.8
%. Northern Africa, the dearest sub-region at 736 EUR/t, combines a large construction part, from
the distance of the central Sahara from towns, with Africa's cost of capital, and so has the largest
part that the two give together.

## Plants at sea

A sea plant's base costs depend on the mean water depth of its cell (`depth_class` in the run's
site-cost table): fixed foundations up to 70 m, floating foundations to 1,000 m, and deep water
beyond. The wind turbine costs about as much as on land, while its foundation, array cables and
development cost 5.5 times as much with fixed foundations and 9 times as much with floating ones; in
deep water the whole turbine costs 3 times as much. The rest of the plant costs 1.72, 2 and 3 times as
much as on land in the three classes. The map shows the classes, with the land cells in grey. The
table gives the median LCOA of the sea plants by depth class for every case; the lines above it
count the sea cells in international waters (no `eez_iso3`), give the cheapest sea plant south of 60
degrees S in each case and count the sea plants of each case that are cheaper than its median land
plant.

In [ ]:
depth = h.depth_classes(SEA_RUNS["base"])
h.plot_maps({"": depth}, "depth_class", title="Depth class of the sea plants", label="Depth class",
            categories=dict(zip(h.DEPTH_CLASSES.values(), h.DEPTH_COLOURS)), background=LAND["base"], background_label="land cells")
plt.show()
print(f"Sea cells in international waters: {depth['eez_iso3'].isna().sum():,} of {len(depth):,}")
print("Cheapest sea plant south of 60 degrees S:",
      ", ".join(f"{case} {SEA[case].loc[SEA[case]['latitude'] <= -60, 'lcoa_eur_per_t'].min():,.0f} EUR/t" for case in RUNS))
print("Sea plants cheaper than the case's median land plant:",
      ", ".join(f"{case} {(SEA[case]['lcoa_eur_per_t'] < LAND[case]['lcoa_eur_per_t'].median()).sum():,}" for case in RUNS))
h.median_by(SEA, depth[["latitude", "longitude", "depth_class"]])

The table lists the ten cheapest sea plants of one case, with the country of the exclusive economic
zone that holds each cell (`eez_iso3`, empty in international waters).

In [ ]:
CASE = "base"  # "base", "site", "finance" or "all"
h.cheapest_cells(SEA[CASE].merge(depth, on=["latitude", "longitude"]), n=10, extra=["eez_iso3", "depth_class"])

In `base`, a sea plant costs 1,069 EUR/t at the median cell, three times as much as a land plant:
746 EUR/t with fixed foundations, 811 EUR/t with floating ones and 1,123 EUR/t in deep water, which
covers 33,136 of the 40,197 sea cells. No sea plant of `base` is cheaper than the median land plant.
The cheapest, at 385 to 411 EUR/t, lie off southern Chile, around the Kerguelen Islands and off the
coast of Antarctica. The model has no data on sea ice, so frozen sea counts as sea, and the cold rule
is a factor on cost, not a limit, which only `site` and `all` apply. The Antarctic cells are therefore
cheap in `base` and `finance`, while in `site` and `all` the cold factor (1.25 beyond 60 degrees of
latitude) and the remoteness factors make the cheapest sea plant south of 60 degrees S cost 1,780 and
2,881 EUR/t.

The cost of capital raises the cost of sea plants more than that of land plants, since 24,254 of the
40,197 sea cells are in international waters, at 11.8 %: the median sea plant of `finance` costs
1,543 EUR/t. Site costs raise it further, because the remoteness factors of a sea plant count the
distance from the cell to the nearest port: the median sea plant of `site` costs 2,582 EUR/t, and that
of `all` 3,777 EUR/t. In `all`, 83 sea plants are cheaper than the median land plant (546 EUR/t); the
cheapest, at 448 to 459 EUR/t, are in the German Bight (`CASE = "all"` lists them).

## Supply curves

A supply curve sorts the plants that can make at least 1 Mt/yr from the cheapest to the dearest and
adds up their production potential (`h.supply_curve`); its height at a production volume is the cost
of the last tonne needed to make that volume. The dashed lines mark the volumes of the summary
table. The table under the figure gives, for the largest volume, the share of the potential of the
plants that make it by country ("no country" where the result table names none, as for most sea
plants).

In [ ]:
SITES = "both"  # "land" for the land plants, "both" for the land and sea plants
XMAX = 10_000  # right end of the curves in Mt/yr; the land plants alone can make about 86,000 Mt/yr
frames = LAND if SITES == "land" else {case: pd.concat([LAND[case], SEA[case]]) for case in RUNS}
h.plot_supply_curves(frames, title=f"Supply curves of the four cases: {'land plants' if SITES == 'land' else 'land and sea plants'}",
                     xmax=XMAX, marks=VOLUMES)
plt.show()
h.supply_shares(frames, VOLUMES[-1], by="country", top=5)

The curves are nearly flat over the first 10,000 Mt/yr, because each of the cheapest cells can make
tens of Mt/yr (the table of the cheapest plants above): in `base`, the cost of the last tonne rises
only from 261 EUR/t at 500 Mt/yr to about 275 EUR/t at 10,000 Mt/yr. At 2,000 Mt/yr, site costs
raise the cost of the last tonne from 271 to 310 EUR/t, finance raises it to 307 EUR/t, and the two
together to 358 EUR/t. Adding the sea plants changes neither volume's cost in any case (summary
table).

The cases also move where the cheapest 2,000 Mt/yr comes from. In `base`, 82 % of it is in Chile,
Bolivia, Argentina and China, mostly on the high plateaus of the Andes and Tibet. The cold and
remoteness factors of `site` move it to lower and less remote sites, in Chile, Namibia, Egypt, Peru,
Oman and Saudi Arabia. The cost of capital of `finance` moves it to countries with low rates and good
sites, 51 % to the United States and 31 % to Australia, while China keeps its share: its rate, 6.6 %,
is close to the 5 % of `base`, and `finance` has no cold factor. In `all`, 56 % is in the United
States, 22 % on the Arabian Peninsula (Saudi Arabia, Oman and the United Arab Emirates) and 4 % in
Spain.

The histograms show the same plants by cost band, 25 EUR/t wide, with the production potential of
each band split by the UN region (M49) of the land plants and with the sea plants as one group; the
box in each panel gives the potential above 2,000 EUR/t, which the bars leave out. A supply curve
shows how much can be made up to a cost; the histograms show where that potential lies. The table
under them gives the potential below 400 EUR/t by group, and that of all the land and all the sea
plants.

In [ ]:
EDGES = range(250, 2001, 25)  # edges of the cost bands in EUR/t; plants above the last edge go in each panel's box
BELOW = 400  # the cost (EUR/t, one of the edges) below which the table sums the potential
tables = {case: h.potential_by_cost(LAND[case], SEA[case], EDGES) for case in RUNS}
h.plot_potential_by_cost(tables, title="Production potential by levelised cost of ammonia: plants that make 1 Mt/yr")
plt.show()
h.potential_below(tables, BELOW)

In `base`, nearly all of the land's 86 Gt/yr costs less than 400 EUR/t, and about 53 Gt/yr lies in
the single band from 275 to 300 EUR/t, more than half of it in Africa and most of the rest in Asia.
Site costs spread that potential over 300 to 750 EUR/t and nearly halve the potential below 400
EUR/t, to 45 Gt/yr. The cost of capital splits it in two: the potential of Asia and Oceania stays
below 400 EUR/t, while Africa's, at 11.8 %, moves to 425 to 500 EUR/t, and none of it remains below
400 EUR/t. In `all`, only 10 Gt/yr remains below 400 EUR/t, mostly in Asia and the Americas, and the
rest of the land's potential spreads up to about 1,250 EUR/t. The sea plants' 124 Gt/yr lies almost
all above 450 EUR/t, and above 2,000 EUR/t for 5 Gt/yr of it in `base` and 105 Gt/yr in `all`.

## Base case with Salmon's weather profiles against ERA5

`base` was also run on the hourly 2019 profiles of Salmon (2022), in campaign `campaign_20261030_v1`,
with the same inputs otherwise, so the change in each cell's LCOA between the two runs comes from the
weather data alone. The map shows that change. The table gives it in bands of wind's share of the
plant's generating capacity with Salmon's profiles (`wind_share_old`, from `h.wind_share`).

In [ ]:
# base on Salmon's profiles: its execution of 30 October 2026 (run_store.csv, notes of its run)
PREVIOUS = ("results/campaigns/campaign_20261030_v1/50_campaign/gl_dea2050_wacc5_bflat_wflat_land20oct_fixed/runs/"
            "20261030T131150Z-cdbdc6788e6d3c34d291ed8234e3b14f807d54d1/global")
LIMITS = (-30, 30)  # ends of the colour scale in per cent
previous = h.read_surface(h.REPO_ROOT / PREVIOUS)
weather = h.compare_surfaces(LAND["base"], previous)
h.plot_maps({"": weather}, "lcoa_change_pct", title="Change in levelised cost of ammonia from Salmon's profiles to ERA5: base, land plants",
            label="Change in levelised cost (%)", scale="change", limits=LIMITS,
            notes={"": h.summary_note(weather["lcoa_change_pct"], "%", signed=True)})
plt.show()
print(f"Cells cheaper with ERA5: {(weather['lcoa_change_pct'] < 0).mean():.0%}")
for name, plants in (("Salmon's profiles", previous), ("ERA5", LAND["base"])):
    print(f"{name}: {(plants['wind_mw'] > 1).mean():.0%} of the plants build wind; all the plants together have "
          f"{plants['wind_mw'].sum() / 1e6:.1f} TW of wind and {plants['solar_mw'].sum() / 1e6:.1f} TW of PV")
BANDS = (0, 0.01, 0.25, 0.5, 1)  # edges of the bands of wind's share of generating capacity with Salmon's profiles
h.by_band(weather, "lcoa_change_pct", "wind_share_old", BANDS)

With ERA5, the LCOA of `base` falls by 4 % at the median cell, and 64 % of the cells are cheaper. The
change follows how much of the plant's generating capacity was wind with Salmon's profiles. Plants
with less than a quarter of their capacity in wind are 6 to 8 % cheaper at the median; plants with a
quarter to a half are 9 % dearer, and plants with more than half are 25 % dearer. On the map, the
dearer cells lie mostly in the windy north of Russia and Canada, in Kazakhstan, Scandinavia,
Patagonia and the Great Plains of the United States, while parts of eastern Siberia, Alaska and the
Yukon, and Mongolia are more than 20 % cheaper. Fewer plants build wind: the share that build any
falls from 68 % to 49 %, and all the plants together have 11.8 TW of wind instead of 17.9 TW, and
89.3 TW of PV instead of 83.4 TW.